# 02 — Train & test

Build a dataset, train the `causal` model (multi-seed), and run the test-gate metrics (per-output regression, calibration, proportion-common-cause vs disparity).



In [2]:
from pathlib import Path
import numpy as np

from causal_msi.config import load_config
from causal_msi.generative import build_dataset
from causal_msi.training import run_multiseed
from causal_msi.utils import seed_everything

cfg = load_config(Path('..') / 'configs' / 'default.yaml')
rng = seed_everything(cfg.seed)
# Use a smaller dataset for a quick notebook run.
dataset = build_dataset(rng, cfg, n_trials=5000)
dataset.X.shape, dataset.Y.shape

((5000, 130), (5000, 5))

In [3]:
results = run_multiseed(cfg, checkpoint_dir='../checkpoints', dataset=dataset)
for r in results:
    print(f'seed {r.seed}: best_val={r.best_val:.4f} @ epoch {r.best_epoch}')

seed 0: best_val=18.9533 @ epoch 199
seed 1: best_val=16.9616 @ epoch 199
seed 2: best_val=16.9453 @ epoch 199
seed 3: best_val=17.4737 @ epoch 199
seed 4: best_val=19.4458 @ epoch 199


In [5]:
# Test-gate: per-output regression decoded-vs-analytical.
import torch
from causal_msi.analysis.performance import per_output_regression

model = results[0].model.eval()
model_device = next(model.parameters()).device
with torch.no_grad():
    input_tensor = torch.as_tensor(dataset.X, dtype=torch.float32).to(model_device)
    pred = model(input_tensor).cpu().numpy()


for reg in per_output_regression(pred, dataset.Y):
    print(reg)

OutputRegression(name='mu_vis', slope=0.7773908190098915, intercept=0.028259155449515534, r2=0.7738515196256135)
OutputRegression(name='var_vis', slope=0.5810914910322008, intercept=4.913943607299614, r2=0.612220054531004)
OutputRegression(name='mu_prop', slope=0.9383855179065458, intercept=0.0018579592402276027, r2=0.9455199542288133)
OutputRegression(name='var_prop', slope=0.8612015070495733, intercept=2.207391937953028, r2=0.8837971847339765)
OutputRegression(name='p_common', slope=0.20779233947598733, intercept=0.39316556594213775, r2=0.25830986582773297)
